# Electron Flow from First Principles
### A Tutorial for Physics Students

In this notebook you will build up — step by step — everything needed to understand **why electrons drift slowly through a wire even though the light turns on instantly**.

By the end you will be able to:
- Calculate the electric field inside a wire from voltage
- Simulate a single electron being accelerated and scattered
- Estimate drift speed from a crowd of electrons
- Plot realistic I–V curves for a copper-like conductor
- Explore how field strength, temperature, and wire geometry affect current

**Run each cell in order.** Read the explanation before running the code.

In [ ]:
# ── Section 1: Imports and Physical Constants ──────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
# %matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 12})

# ── Fundamental constants ──────────────────────────────────────────────────
e      = 1.602e-19   # electron charge magnitude  (C)
m_e    = 9.109e-31   # electron mass              (kg)
k_B    = 1.381e-23   # Boltzmann constant         (J/K)

# ── Copper wire parameters ─────────────────────────────────────────────────
n          = 8.5e28   # free-electron density      (electrons / m³)
mu_0       = 4.4e-3   # electron mobility at 20 °C (m² / V·s)
alpha_Cu   = 3.9e-3   # temperature coefficient    (1 / °C)
rho_0      = 1.7e-8   # resistivity at 20 °C       (Ω·m)
tau_0      = m_e * mu_0 / e   # mean free time at 20 °C (s)

WIRE_L = 60.0    # wire length  (m)
WIRE_A = 1.0e-6  # cross-section (m²)

np.random.seed(42)

print(f"Mean free time  τ₀  = {tau_0:.2e} s")
print(f"Thermal speed (20°C)= {np.sqrt(3*k_B*293/m_e):.2e} m/s  ← fast random motion")
print(f"Wire resistance     = {rho_0*WIRE_L/WIRE_A:.3f} Ω")

---
## Part 1 — Electric Field in a Wire: $E = V/L$

When a battery is connected to a wire, it creates a voltage difference between the two ends.
Inside the wire that voltage drops smoothly from one end to the other — like water pressure dropping along a pipe.

For a uniform wire of length $L$ with voltage $V$ applied:

$$\boxed{E = \frac{V}{L}}$$

A steeper voltage drop → stronger field → harder push on electrons.

In [ ]:
# ── Section 2: Electric field and potential visualisation ──────────────────
L  = 1.0                       # wire length (m) — just for visualisation
V0 = 5.0                       # applied voltage (V)
x  = np.linspace(0, L, 500)

V_linear = V0 * (1 - x / L)   # potential (uniform wire)
E_field  = V0 / L              # constant E field (V/m)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(x * 100, V_linear, color="steelblue", lw=2.5)
ax1.set_xlabel("Position along wire (cm)")
ax1.set_ylabel("Voltage  V(x)  [V]")
ax1.set_title(f"Potential drops from {V0} V to 0 V")
ax1.annotate("Battery +", xy=(0, V0), xytext=(5, V0 - 0.4), fontsize=11, color="red")
ax1.annotate("Battery −", xy=(100, 0), xytext=(70, 0.4), fontsize=11, color="blue")
ax1.grid(alpha=0.3)

ax2.axhline(E_field, color="tomato", lw=2.5, label=f"E = V/L = {V0}/{L} = {E_field:.0f} V/m")
ax2.fill_between(x * 100, 0, E_field, alpha=0.15, color="tomato")
ax2.set_xlabel("Position along wire (cm)")
ax2.set_ylabel("Electric field  E  [V/m]")
ax2.set_title("Field is uniform — same push everywhere")
ax2.set_ylim(0, 8)
ax2.legend(fontsize=12)
ax2.grid(alpha=0.3)

plt.suptitle(f"A {V0} V battery across a {L} m wire", fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 2 — Force on an Electron

Electrons have **negative** charge, so the force on them is **opposite** to the field direction:

$$F = qE = (-e)E \quad \Longrightarrow \quad \text{electrons move toward the + terminal}$$

This is why conventional current (defined as + charge flow) goes the opposite way to electron motion.

In [ ]:
# ── Section 3: Force direction diagram ────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 3))
ax.fill_betweenx([-0.35, 0.35], 0, 100, color="lightyellow", ec="gray", lw=1.5, zorder=0)

for xi in np.linspace(8, 92, 9):
    # Electric field arrow → (red)
    ax.annotate("", xy=(xi + 5, 0.18), xytext=(xi, 0.18),
                arrowprops=dict(arrowstyle="->", color="red", lw=2.2))
    # Electron force arrow ← (blue)
    ax.annotate("", xy=(xi - 5, -0.18), xytext=(xi, -0.18),
                arrowprops=dict(arrowstyle="->", color="royalblue", lw=2.2))

ax.text(50,  0.26, "Electric field  E →", ha="center", color="red",
        fontsize=13, fontweight="bold")
ax.text(50, -0.26, "← Electron drift  (opposite to E)", ha="center",
        color="royalblue", fontsize=13, fontweight="bold")
ax.text( 1,  0, "−", fontsize=24, color="blue",  va="center", fontweight="bold")
ax.text(96,  0, "+", fontsize=24, color="red",   va="center", fontweight="bold")
ax.set_xlim(0, 100); ax.set_ylim(-0.5, 0.5)
ax.set_xlabel("Position along wire (cm)"); ax.set_yticks([])
ax.set_title("Electrons move OPPOSITE to the electric field")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 3 — Simulating a Single Electron (No Collisions)

Applying Newton's second law to one electron:

$$m_e \frac{dv}{dt} = F = -eE$$

We use **Euler integration** — tiny time steps — to track where the electron goes.
Without anything stopping it, the electron accelerates indefinitely. Watch what happens below,
then in Part 4 we add the collisions that limit the speed.

In [ ]:
# ── Section 4: Single electron — free acceleration ─────────────────────────
E_sim  = 5.0    # V/m
dt     = 1e-15  # time step (s)
N      = 3000

x_e = np.zeros(N); v_e = np.zeros(N)
t   = np.arange(N) * dt

for i in range(1, N):
    v_e[i] = v_e[i-1] + (-e * E_sim / m_e) * dt
    x_e[i] = x_e[i-1] + v_e[i-1] * dt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax1.plot(t * 1e15, x_e * 1e9, color="steelblue", lw=2)
ax1.set_ylabel("Position  (nm)"); ax1.set_title("No collisions — electron accelerates freely")
ax1.grid(alpha=0.3)
ax2.plot(t * 1e15, v_e / 1e3, color="tomato", lw=2)
ax2.set_ylabel("Speed  (km/s)"); ax2.set_xlabel("Time  (fs = 10⁻¹⁵ s)")
ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("↑ Electron just keeps speeding up — not physical. We need collisions!")

---
## Part 4 — Adding Collisions: The Drude Model (~1900)

In a real metal, electrons collide with vibrating lattice ions roughly every **25 femtoseconds**.
After each collision the electron's velocity is randomised — it loses its momentum.

The average time between collisions is the **mean free time** $\tau$.
Each time step, the probability of a collision is $P = dt/\tau$.

After a collision, the electron gets a new **random thermal velocity** — fast but in a random direction.
The field slowly builds up a tiny average drift. That average is the **drift speed** $v_d$.

In [ ]:
# ── Section 5: Single electron WITH scattering ────────────────────────────
tau       = tau_0                                     # mean free time
v_th      = np.sqrt(3 * k_B * 293 / m_e)             # thermal speed (m/s)
E_sim     = 5.0                                       # V/m
dt        = 1e-16                                     # smaller step
N         = 60_000
p_coll    = dt / tau                                  # collision probability per step

x_s = np.zeros(N); v_s = np.zeros(N)
v = 0.0; x = 0.0
t = np.arange(N) * dt

for i in range(1, N):
    v += (-e * E_sim / m_e) * dt                      # accelerate
    x += v * dt
    if np.random.random() < p_coll:                   # collision?
        v = np.random.normal(0, v_th / np.sqrt(3))   # random thermal kick
    v_s[i] = v; x_s[i] = x

mask = t < 3e-13   # first 300 fs
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

ax1.plot(t[mask] * 1e15, x_s[mask] * 1e9, color="seagreen", lw=1.2)
ax1.set_ylabel("Position  (nm)")
ax1.set_title("Single electron WITH collisions (Drude model)")
ax1.grid(alpha=0.3)

ax2.plot(t[mask] * 1e15, v_s[mask] / 1e3, color="seagreen", lw=0.8, alpha=0.7)
mean_v = v_s[mask].mean()
ax2.axhline(mean_v / 1e3, color="red", lw=2, ls="--",
            label=f"Mean velocity = {mean_v:.4f} m/s  (the drift speed!)")
ax2.set_ylabel("Velocity  (km/s)"); ax2.set_xlabel("Time  (fs)")
ax2.legend(fontsize=11); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Thermal speed:      {v_th/1e3:.1f} km/s  (fast random motion)")
print(f"Mean drift speed:   {abs(mean_v)*1e3:.4f} mm/s  (tiny net drift)")

---
## Part 5 — Drift Speed from Many Electrons

One electron is noisy. With **many electrons** the random thermal velocities average to zero
and the leftover is the **drift velocity** $v_d$.

The Drude model predicts:

$$v_d = \frac{eE\tau}{m_e} = \mu E \qquad \text{where } \mu = \frac{e\tau}{m_e} \text{ is the electron mobility}$$

We run a Monte Carlo simulation with 200 electrons to verify this.

In [ ]:
# ── Section 6: Ensemble drift — many electrons ────────────────────────────
N_e    = 200
E_sim  = 5.0
dt     = 1e-16
N      = 40_000
p_coll = dt / tau_0
v_th   = np.sqrt(3 * k_B * 293 / m_e)

vels    = np.random.normal(0, v_th / np.sqrt(3), N_e)   # start with thermal spread
mean_v  = np.zeros(N)
t       = np.arange(N) * dt

for i in range(N):
    vels    += (-e * E_sim / m_e) * dt
    hit      = np.random.random(N_e) < p_coll
    vels[hit] = np.random.normal(0, v_th / np.sqrt(3), hit.sum())
    mean_v[i] = vels.mean()

v_d_theory = e * E_sim * tau_0 / m_e   # Drude prediction

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(t * 1e15, mean_v, color="steelblue", lw=1, alpha=0.8, label=f"{N_e} electrons")
ax1.axhline(v_d_theory, color="red", lw=2, ls="--", label=f"Drude: v_d = μE = {v_d_theory:.4f} m/s")
ax1.axhline(0, color="gray", ls=":", lw=1)
ax1.set_xlabel("Time (fs)"); ax1.set_ylabel("Mean velocity (m/s)")
ax1.set_title("Mean drift converges to Drude prediction")
ax1.legend(); ax1.grid(alpha=0.3)

running_avg = np.cumsum(mean_v) / (np.arange(N) + 1)
ax2.plot(t * 1e15, running_avg, color="seagreen", lw=1.5, label="Running average")
ax2.axhline(v_d_theory, color="red", lw=2, ls="--", label=f"Theory: {v_d_theory:.5f} m/s")
ax2.set_xlabel("Time (fs)"); ax2.set_ylabel("Running mean velocity (m/s)")
ax2.set_title("Time-average → drift speed")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Drude theory:    v_d = eEτ/m = {v_d_theory:.5f} m/s")
print(f"Simulation avg:  v_d         = {mean_v[-5000:].mean():.5f} m/s")

---
## Part 6 — Current Density and Ohm's Law

Now that we have drift speed, we can find **current density** — the current per unit area:

$$J = nev_d = ne\mu E = \sigma E \quad \left[\frac{\text{A}}{\text{m}^2}\right]$$

where $\sigma = ne\mu$ is the **electrical conductivity**.  
Total current through a wire of cross-section $A$:

$$I = JA = neAv_d$$

Since $v_d = \mu E = \mu V/L$, and $R = L/(\sigma A)$, this gives $I = V/R$ — **Ohm's Law!**

In [ ]:
# ── Section 7: Drift speed and current density vs E field ─────────────────
E_range  = np.linspace(0, 20, 300)
v_d_arr  = mu_0 * E_range           # drift speed (m/s)
J_arr    = n * e * v_d_arr          # current density (A/m²)
sigma    = n * e * mu_0

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

ax1.plot(E_range, v_d_arr * 1e3, color="steelblue", lw=2.5)
ax1.set_xlabel("Electric field E  (V/m)")
ax1.set_ylabel("Drift speed  (mm/s)")
ax1.set_title("$v_d = \\mu E$  — linear!")
ax1.grid(alpha=0.3)

ax2.plot(E_range, J_arr / 1e6, color="tomato", lw=2.5)
ax2.set_xlabel("Electric field E  (V/m)")
ax2.set_ylabel("Current density  (MA/m²)")
ax2.set_title("$J = nev_d = \\sigma E$  — Ohm's Law")
ax2.grid(alpha=0.3)

# Thermal vs drift speed comparison at E = 5 V/m
labels = ["Thermal speed\n(random)", "Drift speed\n(net)"]
values = [np.sqrt(3*k_B*293/m_e)/1e3, mu_0 * 5.0 * 1e3]
colors = ["tomato", "steelblue"]
bars   = ax3.bar(labels, values, color=colors, edgecolor="white", linewidth=1.5)
ax3.set_ylabel("Speed")
ax3.set_title(f"Speed comparison at E = 5 V/m\n(different units — note the scale!)")
for bar, val, unit in zip(bars, values, ["km/s", "mm/s"]):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 0.5,
             f"{val:.0f} {unit}", ha="center", va="center",
             color="white", fontsize=12, fontweight="bold")
ax3.grid(axis="y", alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Conductivity σ = {sigma:.2e} S/m  (copper actual: ~6×10⁷ S/m)")

---
## Part 7 — Temperature Effects and I–V Curves

Higher temperature → ions vibrate more → electrons collide more often → shorter $\tau$ → lower mobility:

$$\mu(T) = \frac{\mu_0}{1 + \alpha(T - 20°C)} \qquad \rho(T) = \rho_0\,(1 + \alpha\Delta T)$$

An **Ohmic** conductor (constant temperature) gives a straight I–V line.  
A **self-heating** wire curves downward — the same effect that makes a light-bulb filament non-linear.

In [ ]:
# ── Section 8: Temperature effects and I-V curves ─────────────────────────
V_range = np.linspace(0, 10, 300)
R_20    = rho_0 * WIRE_L / WIRE_A      # resistance at 20°C

# Ohmic (fixed temperature)
I_ohmic = V_range / R_20

# Temperature effect: v_d vs T at fixed voltage
T_range  = np.linspace(0, 300, 300)
mu_T     = mu_0 / (1 + alpha_Cu * (T_range - 20))
v_d_T    = mu_T * (3.4 / WIRE_L)      # drift speed at 3.4 V (mm/s)

# Self-heating non-Ohmic
k_heat   = 2.0    # °C/A²
I_nonohmic = np.zeros_like(V_range)
for idx, V in enumerate(V_range):
    I_g = V / R_20
    for _ in range(40):
        T_wire = 20 + k_heat * I_g**2
        R_T    = rho_0 * (1 + alpha_Cu * (T_wire - 20)) * WIRE_L / WIRE_A
        I_new  = V / R_T
        if abs(I_new - I_g) < 1e-7: break
        I_g    = 0.5 * I_g + 0.5 * I_new
    I_nonohmic[idx] = I_new

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(T_range, v_d_T * 1e6, color="steelblue", lw=2.5)
ax1.set_xlabel("Temperature  (°C)")
ax1.set_ylabel("Drift speed  (μm/s)")
ax1.set_title("Drift speed decreases with temperature\n(at V = 3.4 V, 60 m copper wire)")
ax1.axvline(20,  color="gray", ls=":", label="Room temp (20°C)")
ax1.axvline(300, color="tomato", ls=":", alpha=0.7, label="Max slider (300°C)")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(V_range, I_ohmic,    color="steelblue", lw=2.5, label="Ohmic (constant T)")
ax2.plot(V_range, I_nonohmic, color="tomato",    lw=2.5, ls="--", label="Self-heating (non-Ohmic)")
ax2.set_xlabel("Voltage  (V)"); ax2.set_ylabel("Current  (A)")
ax2.set_title("I–V curves: Ohmic vs Self-Heating")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Resistance at 20°C:  {R_20:.3f} Ω")
print(f"Resistance at 300°C: {rho_0*(1+alpha_Cu*280)*WIRE_L/WIRE_A:.3f} Ω  (~{1+alpha_Cu*280:.1f}× higher)")

---
## Part 8 — Interactive Exploration

Use the sliders to see how each parameter changes drift speed, current, and the I–V curve in real time.

| Slider | What it controls |
|--------|-----------------|
| **Voltage** | Applied potential difference (V) |
| **Temperature** | Wire temperature → changes $\tau$ and mobility |
| **Carrier density** | More free electrons per m³ |
| **Mean free time (× τ₀)** | Shorter → more collisions → lower drift |

In [ ]:
# ── Section 9: Interactive widget dashboard ────────────────────────────────
from ipywidgets import interact, FloatSlider

def drift_dashboard(voltage=3.4, temperature=20.0, log_n=28.93, tau_factor=1.0):
    mu      = (mu_0 * tau_factor) / (1 + alpha_Cu * (temperature - 20))
    E       = voltage / WIRE_L
    v_d     = mu * E                        # m/s
    n_val   = 10**log_n
    J       = n_val * e * v_d               # A/m²
    I       = J * WIRE_A                    # A
    rho_T   = rho_0 / tau_factor * (1 + alpha_Cu * (temperature - 20))
    R_T     = rho_T * WIRE_L / WIRE_A

    Vs   = np.linspace(0, 10, 200)
    Is   = Vs / R_T

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Panel 1: drift speed bar with reference
    ax = axes[0]
    ax.barh(["Your settings"], [v_d * 1e6], color="steelblue", height=0.35)
    ax.barh(["Reference\n(3.4V, 20°C)"], [mu_0 * 3.4/WIRE_L * 1e6],
            color="gray", height=0.35, alpha=0.5)
    ax.set_xlabel("Drift speed  (μm/s)"); ax.set_xlim(0, 900)
    ax.set_title(f"v_d = {v_d*1e6:.1f}  μm/s")
    ax.grid(axis="x", alpha=0.3)

    # Panel 2: thermal vs drift comparison
    ax = axes[1]
    v_th_now = np.sqrt(3 * k_B * (temperature + 273) / m_e)
    ax.bar(["Thermal\n(km/s)", "Drift\n(μm/s)"],
           [v_th_now/1e3, v_d*1e6],
           color=["tomato", "steelblue"], edgecolor="white")
    ax.set_title("Thermal vs Drift speed\n(completely different scales!)")
    ax.grid(axis="y", alpha=0.3)

    # Panel 3: I-V curve with operating point
    ax = axes[2]
    ax.plot(Vs, Is, color="seagreen", lw=2.5, label=f"T={temperature:.0f}°C, τ={tau_factor:.1f}τ₀")
    ax.scatter([voltage], [I], color="red", s=150, zorder=5,
               label=f"V={voltage:.1f}V → I={I:.2f}A")
    ax.set_xlabel("Voltage (V)"); ax.set_ylabel("Current (A)")
    ax.set_title("I–V Curve")
    ax.legend(fontsize=10); ax.grid(alpha=0.3); ax.set_xlim(0, 10); ax.set_ylim(0)

    plt.suptitle(
        f"E = {E:.4f} V/m  |  μ = {mu:.4e} m²/V·s  |  v_d = {v_d*1e6:.1f} μm/s  |  I = {I:.2f} A",
        fontsize=11, y=1.02
    )
    plt.tight_layout(); plt.show()

interact(
    drift_dashboard,
    voltage     = FloatSlider(min=0, max=10,  step=0.1,  value=3.4,  description="Voltage (V)",   style={"description_width": "120px"}),
    temperature = FloatSlider(min=0, max=300, step=5,    value=20,   description="Temp (°C)",     style={"description_width": "120px"}),
    log_n       = FloatSlider(min=27, max=30, step=0.05, value=28.93,description="log₁₀(n)",      style={"description_width": "120px"}),
    tau_factor  = FloatSlider(min=0.1, max=3, step=0.1,  value=1.0,  description="τ / τ₀",        style={"description_width": "120px"}),
);

---
## Summary — The Full Chain

| Step | Equation | Meaning |
|------|----------|---------|
| 1 | $E = V / L$ | Voltage spread across wire length gives the field |
| 2 | $F = -eE$ | Field pushes electrons opposite to its own direction |
| 3 | $\tau = m_e\mu_0 / e$ | Mean free time set by the material |
| 4 | $v_d = \mu E = (e\tau/m_e)\,E$ | Drift speed: mobility × field |
| 5 | $\mu(T) = \mu_0\,/\,(1+\alpha\Delta T)$ | Higher temperature → more collisions → lower mobility |
| 6 | $J = nev_d = \sigma E$ | Current density: Ohm's Law at the microscopic level |
| 7 | $I = JA = neAv_d$ | Total current in a wire of cross-section $A$ |

### Connection to `drift.py`

In the VPython simulation you fill in two lines:

```python
def compute_drift_speed(voltage, wire_length, mobility):
    E   = None    # → voltage / wire_length      (step 1)
    v_d = None    # → mobility * E               (step 4)
    return v_d
```

Everything else — computing $\mu(T)$, current, resistance — is already handled for you.